# LaRS Dataset Embedding Analysis

## Dataset Overview

The **LaRS** (Lake/River/Sea) dataset is used in the MaCVi 2026 embedded semantic-segmentation
challenge track. It covers maritime scenes with three semantic classes: water, obstacles, and sky.

| Split | Images | Notes |
|-------|--------|-------|
| train | 2,605  | Labelled images |
| val   |   198  | Labelled images |
| test  | 1,203  | Labels withheld |
| **total** | **4,006** | |

Typical resolutions: 1278 × 958 (landscape) and 1280 × 720 (widescreen HD).

## Model Choice: NVIDIA C-RADIOv3-H

**RADIO** (Radio Amplitude and Distortion Importance Optimization) is a vision foundation model
trained by NVIDIA as a multi-teacher distillation of DINOv2, SAM, CLIP, and other experts.
RADIO-H is the largest variant (ViT-H backbone, ~632 M parameters).

Reasons for choosing RADIO over other encoders:

- **Spatial richness**: RADIO retains full spatial token grids, not just a [CLS] token,
  giving per-region sensitivity that is critical for detecting small maritime obstacles.
- **Resolution flexibility**:  snaps images to a valid
  resolution, avoiding padding artifacts when working with non-standard aspect ratios.
- **No explicit crop extraction**: Instead of manually cropping regions of interest, we let
  the model attend to the whole scene. The spatial token average is a compact, semantically
  rich descriptor. This is more robust to unknown object positions than crop-based methods.
- **Strong OOD sensitivity**: RADIO was trained on diverse web-scale data, so its embedding
  space reliably reflects visual distribution shifts — exactly what we need for OOD detection.

## Resolution Choice

Images are processed at full original resolution, snapped to the nearest resolution supported
by RADIO (via ). This avoids quality loss from downscaling
while keeping inference stable.

## What Each Section Shows

| Section | Content | Why |
|---------|---------|-----|
| 2 | Embedding extraction | Slow; runs once, writes to SQLite |
| 3 | Load embeddings | Fast; describe split/sequence statistics |
| 4 | t-SNE computation | Medium; runs once, saved to .npy |
| 5 | t-SNE by split | Check if train/val/test overlap |
| 6 | t-SNE by sequence | Detect sequence-level clusters or outliers |
| 7 | Cosine similarity (OOD) | Quantify how well training data covers test |
| 8 | Report generation | Export to PDF |
| 9 | Summary table | One-page overview |

## Interpretation Guide

- **t-SNE clusters**: Tight clusters by split indicate good coverage; isolated test points
  suggest scenes not represented in training.
- **Max cosine similarity**: For each test image we find the most similar training image.
  Low max-sim (< 0.70) indicates that no training image is close in embedding space — a
  potential OOD sample that may be harder for a segmentation model.
- **Sequence colouring**: If a sequence forms a tight cluster far from training data,
  consider adding augmentation or collecting more training data for that environment.


## Section 1 — Setup & Configuration

In [ ]:
import os
import sys
import sqlite3
import base64
import io
import time
import logging
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image

import torch
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---- Paths ----
ANALYSIS_DIR = Path("/home/farms/macvi2026-farms-team/challenges/common/lars-analysis")
DATASET_ROOT = Path("/home/farms/data/lars_dataset/img_dir")
DB_PATH      = ANALYSIS_DIR / "embeddings.db"
TSNE_PATH    = ANALYSIS_DIR / "tsne_results.npy"
REPORT_PATH  = ANALYSIS_DIR / "report.pdf"
LOG_PATH     = ANALYSIS_DIR / "logs" / "analysis.log"

# ---- Model config ----
MODEL_NAME_H = "nvidia/C-RADIOv3-H"
MODEL_NAME_L = "nvidia/C-RADIOv3-L"
BATCH_SIZE   = 1          # RADIO-H is large; process one image at a time
COMMIT_EVERY = 50         # SQLite commit frequency

# ---- VRAM check ----
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        total_gb = props.total_memory / 1024**3
        alloc_gb = torch.cuda.memory_allocated(i) / 1024**3
        print(f"  GPU {i}: {props.name}  total={total_gb:.1f} GB  alloc={alloc_gb:.2f} GB")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


## Section 2 — Embedding Extraction

**Slow cell — skipped if  already contains all 4 006 rows.**

In [ ]:
# ---- helpers ----

def get_image_list(split: str):
    """Return list of (img_name, split, sequence, img_path) for a split."""
    list_file = DATASET_ROOT / split / "image_list.txt"
    names = list_file.read_text().strip().splitlines()
    records = []
    for name in names:
        parts = name.rsplit("_", 1)
        sequence = parts[0] if len(parts) > 1 else name
        img_path = DATASET_ROOT / split / "images" / f"{name}.jpg"
        records.append((name, split, sequence, img_path))
    return records


def init_db(conn):
    cur = conn.cursor()
    cur.executescript("""
        CREATE TABLE IF NOT EXISTS metadata (
            key   TEXT PRIMARY KEY,
            value TEXT
        );
        CREATE TABLE IF NOT EXISTS embeddings (
            id                    INTEGER PRIMARY KEY AUTOINCREMENT,
            img_name              TEXT    NOT NULL,
            split                 TEXT    NOT NULL,
            sequence              TEXT    NOT NULL,
            width                 INTEGER,
            height                INTEGER,
            summary_embedding     BLOB,
            spatial_mean_embedding BLOB
        );
        CREATE INDEX IF NOT EXISTS idx_split    ON embeddings(split);
        CREATE INDEX IF NOT EXISTS idx_sequence ON embeddings(sequence);
    """)
    conn.commit()


def get_already_done(conn):
    cur = conn.cursor()
    cur.execute("SELECT img_name FROM embeddings")
    return {row[0] for row in cur.fetchall()}


def embedding_to_blob(arr: np.ndarray) -> bytes:
    return arr.astype(np.float32).tobytes()


# ---- load / skip check ----
TOTAL_IMAGES = 4006

def count_done():
    if not DB_PATH.exists():
        return 0
    with sqlite3.connect(DB_PATH) as c:
        return c.execute("SELECT COUNT(*) FROM embeddings").fetchone()[0]


if count_done() >= TOTAL_IMAGES:
    print(f"embeddings.db already has {count_done()} rows — skipping extraction.")
else:
    print(f"Starting extraction ({count_done()}/{TOTAL_IMAGES} already done)...")

    # --- Load model ---
    from transformers import AutoModel
    import gc

    model_name = MODEL_NAME_H
    try:
        print(f"Loading {model_name} ...")
        radio_model = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        radio_model = radio_model.to(DEVICE).eval()
        # Quick OOM probe
        with torch.no_grad(), torch.autocast(DEVICE, dtype=torch.bfloat16):
            probe = torch.zeros(1, 3, 512, 512, device=DEVICE)
            _ = radio_model(probe)
        del probe
        torch.cuda.empty_cache()
        print(f"Loaded {model_name} successfully.")
    except torch.cuda.OutOfMemoryError:
        print(f"OOM with {model_name}, falling back to {MODEL_NAME_L}")
        model_name = MODEL_NAME_L
        del radio_model
        gc.collect(); torch.cuda.empty_cache()
        radio_model = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        radio_model = radio_model.to(DEVICE).eval()
        print(f"Loaded {model_name} successfully (fallback).")

    ACTUAL_MODEL = model_name
    print(f"Model used: {ACTUAL_MODEL}")

    # --- Gather all records ---
    all_records = []
    for split in ("train", "val", "test"):
        all_records.extend(get_image_list(split))
    print(f"Total records: {len(all_records)}")

    # --- Open DB ---
    conn = sqlite3.connect(DB_PATH)
    init_db(conn)
    already_done = get_already_done(conn)

    # Write metadata
    conn.execute("INSERT OR IGNORE INTO metadata VALUES (?,?)", ("model_name", ACTUAL_MODEL))
    conn.execute("INSERT OR IGNORE INTO metadata VALUES (?,?)", ("created_at", time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())))
    conn.commit()

    pending = [(n, sp, seq, p) for n, sp, seq, p in all_records if n not in already_done]
    print(f"Pending: {len(pending)} images")

    # --- Extract ---
    embed_dim = None
    batch_data = []

    def flush_batch(batch_data, conn):
        conn.executemany(
            "INSERT INTO embeddings (img_name, split, sequence, width, height, summary_embedding, spatial_mean_embedding) "
            "VALUES (?,?,?,?,?,?,?)",
            batch_data,
        )
        conn.commit()


    with tqdm(total=len(pending), desc="Extracting embeddings") as pbar:
        for i, (img_name, split, sequence, img_path) in enumerate(pending):
            try:
                img_pil = Image.open(img_path).convert("RGB")
                W, H = img_pil.size

                # Snap to nearest supported resolution
                th, tw = radio_model.get_nearest_supported_resolution(H, W)
                img_resized = img_pil.resize((tw, th), Image.BICUBIC)

                # Tensor
                img_t = torch.from_numpy(
                    np.array(img_resized).astype(np.float32) / 255.0
                ).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

                with torch.no_grad(), torch.autocast(DEVICE, dtype=torch.bfloat16):
                    output = radio_model(img_t)

                # Handle tuple or named output
                if isinstance(output, (list, tuple)) and len(output) >= 2:
                    summary_t, spatial_t = output[0], output[1]
                elif hasattr(output, "summary") and hasattr(output, "spatial_features"):
                    summary_t  = output.summary
                    spatial_t  = output.spatial_features
                else:
                    # Fallback: treat entire output as spatial
                    spatial_t = output
                    summary_t = spatial_t.mean(dim=1) if spatial_t.ndim == 3 else spatial_t

                # [1, D] and [1, T, D]
                summary_vec = summary_t.squeeze(0).float().cpu().numpy()  # (D,)
                if spatial_t.ndim == 3:
                    spatial_mean = spatial_t.squeeze(0).float().cpu().numpy().mean(axis=0)  # (D,)
                else:
                    spatial_mean = spatial_t.squeeze(0).float().cpu().numpy()

                if embed_dim is None:
                    embed_dim = summary_vec.shape[0]
                    conn.execute("INSERT OR IGNORE INTO metadata VALUES (?,?)", ("embed_dim", str(embed_dim)))
                    conn.commit()

                batch_data.append((
                    img_name, split, sequence, W, H,
                    embedding_to_blob(summary_vec),
                    embedding_to_blob(spatial_mean),
                ))

                if len(batch_data) >= COMMIT_EVERY:
                    flush_batch(batch_data, conn)
                    batch_data = []

            except Exception as e:
                print(f"
ERROR on {img_name}: {e}")

            pbar.update(1)
            pbar.set_postfix({"processed": f"{i+1}/{len(pending)}"})

    if batch_data:
        flush_batch(batch_data, conn)

    conn.close()

    # Update metadata
    with sqlite3.connect(DB_PATH) as c:
        if embed_dim is not None:
            c.execute("INSERT OR REPLACE INTO metadata VALUES (?,?)", ("embed_dim", str(embed_dim)))
            c.execute("INSERT OR REPLACE INTO metadata VALUES (?,?)", ("model_name", ACTUAL_MODEL))
        c.commit()

    n_done = count_done()
    print(f"Extraction complete: {n_done}/{TOTAL_IMAGES} embeddings in DB")


## Section 3 — Load Embeddings

In [ ]:
import sqlite3
import numpy as np
import pandas as pd

def load_embeddings(db_path):
    con = sqlite3.connect(db_path)
    cur = con.execute(
        "SELECT id, img_name, split, sequence, width, height, "
        "summary_embedding, spatial_mean_embedding FROM embeddings ORDER BY id"
    )
    rows = cur.fetchall()
    con.close()

    records = []
    summary_list = []
    spatial_list = []
    for row in rows:
        _id, img_name, split, seq, w, h, sum_blob, spa_blob = row
        records.append({
            "id": _id, "img_name": img_name, "split": split,
            "sequence": seq, "width": w, "height": h,
        })
        summary_list.append(np.frombuffer(sum_blob, dtype=np.float32).copy())
        spatial_list.append(np.frombuffer(spa_blob, dtype=np.float32).copy())

    df = pd.DataFrame(records)
    summary_arr = np.stack(summary_list)   # (N, D)
    spatial_arr = np.stack(spatial_list)   # (N, D)
    return df, summary_arr, spatial_arr


# --- Load metadata ---
def load_metadata(db_path):
    con = sqlite3.connect(db_path)
    meta = dict(con.execute("SELECT key, value FROM metadata").fetchall())
    con.close()
    return meta


meta = load_metadata(DB_PATH)
print("Database metadata:")
for k, v in meta.items():
    print(f"  {k}: {v}")

df, summary_arr, spatial_arr = load_embeddings(DB_PATH)
print(f"
Loaded {len(df)} embeddings, embed_dim={spatial_arr.shape[1]}")

# --- Stats table ---
print("
=== Count per split ===")
print(df.groupby("split").size().reset_index(name="n_images").to_string(index=False))

print("
=== Sequences per split ===")
print(df.groupby("split")["sequence"].nunique().reset_index(name="n_sequences").to_string(index=False))

print("
=== Top-10 sequences by image count ===")
print(df.groupby("sequence").size().sort_values(ascending=False).head(10).reset_index(name="n_images").to_string(index=False))


## Section 4 — t-SNE

**Medium cell — skipped if  already exists.**

In [ ]:
from pathlib import Path

if TSNE_PATH.exists():
    tsne_result = np.load(TSNE_PATH)
    print(f"Loaded t-SNE from {TSNE_PATH}, shape={tsne_result.shape}")
else:
    print("Computing t-SNE ...")
    X = spatial_arr  # (N, D)

    # Step 1: PCA to 50 dimensions
    from sklearn.decomposition import PCA
    n_pca = min(50, X.shape[1], X.shape[0] - 1)
    print(f"PCA: {X.shape[1]} -> {n_pca} dims")
    pca = PCA(n_components=n_pca, random_state=42)
    X_pca = pca.fit_transform(X)
    print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.3f}")

    # Step 2: t-SNE to 2D — try cuML first
    tsne_backend = "unknown"
    try:
        from cuml.manifold import TSNE as cumlTSNE
        import cudf
        print("Using cuML t-SNE (GPU)")
        tsne_backend = "cuml"
        tsne = cumlTSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42, verbose=True)
        X_pca_gdf = cudf.DataFrame(X_pca.astype(np.float32))
        tsne_result = tsne.fit_transform(X_pca_gdf).to_numpy()
    except Exception as e:
        print(f"cuML t-SNE failed ({e}); falling back to sklearn")
        tsne_backend = "sklearn"
        from sklearn.manifold import TSNE
        tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42,
                    n_jobs=-1, verbose=2)
        tsne_result = tsne.fit_transform(X_pca)

    np.save(TSNE_PATH, tsne_result)
    print(f"t-SNE done (backend={tsne_backend}), saved to {TSNE_PATH}")

df["tsne_0"] = tsne_result[:, 0]
df["tsne_1"] = tsne_result[:, 1]
print("t-SNE coordinates added to DataFrame.")


## Section 5 — t-SNE Coloured by Split

In [ ]:
# Build thumbnail cache (base64 PNG, downsized to 64x64)
def make_thumb_b64(img_path, size=(64, 64)):
    try:
        img = Image.open(img_path).convert("RGB")
        img.thumbnail(size, Image.LANCZOS)
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()
    except Exception:
        return ""


print("Building thumbnail cache (this may take ~2 min for 4006 images)...")
thumbs = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Thumbnails"):
    p = DATASET_ROOT / row["split"] / "images" / f"{row['img_name']}.jpg"
    thumbs.append(make_thumb_b64(p))
df["thumb"] = thumbs
print("Done.")

SPLIT_COLORS = {"train": "#1f77b4", "val": "#ff7f0e", "test": "#2ca02c"}

fig5 = go.Figure()
for split in ["train", "val", "test"]:
    mask = df["split"] == split
    sub = df[mask]
    fig5.add_trace(go.Scatter(
        x=sub["tsne_0"], y=sub["tsne_1"],
        mode="markers",
        name=split,
        marker=dict(size=4, color=SPLIT_COLORS[split], opacity=0.7),
        customdata=sub[["img_name", "sequence", "thumb"]].values,
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "seq: %{customdata[1]}<br>"
            "<img src='%{customdata[2]}' width=64 height=64><extra></extra>"
        ),
    ))

fig5.update_layout(
    title="t-SNE of LaRS embeddings — coloured by split",
    xaxis_title="t-SNE dim 0",
    yaxis_title="t-SNE dim 1",
    width=900, height=650,
    legend_title="Split",
)
fig5.show()


## Section 6 — t-SNE Coloured by Sequence

In [ ]:
# Sequences with < 5 images grouped as "other"
seq_counts = df["sequence"].value_counts()
major_seqs = seq_counts[seq_counts >= 5].index.tolist()
df["seq_label"] = df["sequence"].where(df["sequence"].isin(major_seqs), other="other")

print(f"Major sequences (>=5 images): {len(major_seqs)}")
print(f"Images grouped as other: {(df['seq_label']=='other').sum()}")

# Use a qualitative palette
import plotly.express as px
all_labels = sorted([s for s in df["seq_label"].unique() if s != "other"]) + ["other"]
palette = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24 + px.colors.qualitative.Light24
color_map = {label: palette[i % len(palette)] for i, label in enumerate(all_labels)}
color_map["other"] = "#aaaaaa"

fig6 = go.Figure()
for label in all_labels:
    mask = df["seq_label"] == label
    sub = df[mask]
    fig6.add_trace(go.Scatter(
        x=sub["tsne_0"], y=sub["tsne_1"],
        mode="markers",
        name=label,
        marker=dict(size=4, color=color_map[label], opacity=0.7),
        customdata=sub[["img_name", "split", "thumb"]].values,
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "split: %{customdata[1]}<br>"
            "<img src='%{customdata[2]}' width=64 height=64><extra></extra>"
        ),
    ))

fig6.update_layout(
    title="t-SNE of LaRS embeddings — coloured by sequence",
    xaxis_title="t-SNE dim 0",
    yaxis_title="t-SNE dim 1",
    width=1200, height=700,
    legend_title="Sequence",
    showlegend=(len(all_labels) <= 60),  # hide legend if too many
)
fig6.show()


## Section 7 — Cosine Similarity Distribution (OOD Detection)

In [ ]:
from sklearn.preprocessing import normalize as sk_normalize

# Normalise all spatial embeddings to unit length for cosine similarity
spatial_norm = sk_normalize(spatial_arr, norm="l2")  # (N, D)

train_mask = (df["split"] == "train").values
val_mask   = (df["split"] == "val").values
test_mask  = (df["split"] == "test").values

train_emb = spatial_norm[train_mask]  # (2605, D)
val_emb   = spatial_norm[val_mask]    # (198,  D)
test_emb  = spatial_norm[test_mask]   # (1203, D)

print(f"Computing max cosine similarity for {val_emb.shape[0]} val + {test_emb.shape[0]} test images vs {train_emb.shape[0]} train...")

def max_cosine_sim(query_emb, ref_emb, chunk=200):
    """Compute row-wise max cosine similarity (query vs ref), chunked for memory."""
    results = []
    for i in tqdm(range(0, len(query_emb), chunk), desc="Cosine sim"):
        batch = query_emb[i:i+chunk]        # (chunk, D)
        sims  = batch @ ref_emb.T           # (chunk, N_train)
        results.append(sims.max(axis=1))    # (chunk,)
    return np.concatenate(results)

max_sim_val  = max_cosine_sim(val_emb,  train_emb)
max_sim_test = max_cosine_sim(test_emb, train_emb)

# Attach back to df
df.loc[val_mask,  "max_train_sim"] = max_sim_val
df.loc[test_mask, "max_train_sim"] = max_sim_test

# ---- histogram ----
fig7 = go.Figure()
fig7.add_trace(go.Histogram(
    x=max_sim_val, name="val",
    opacity=0.6, nbinsx=50,
    marker_color="#ff7f0e",
))
fig7.add_trace(go.Histogram(
    x=max_sim_test, name="test",
    opacity=0.6, nbinsx=50,
    marker_color="#2ca02c",
))
fig7.add_vline(x=0.5, line_dash="dash", line_color="red",  annotation_text="0.5")
fig7.add_vline(x=0.7, line_dash="dash", line_color="gray", annotation_text="0.7")
fig7.update_layout(
    barmode="overlay",
    title="Max cosine similarity to training set",
    xaxis_title="Max cosine similarity",
    yaxis_title="Count",
    width=850, height=500,
)
fig7.show()

# ---- statistics ----
OOD_THRESHOLDS = [0.5, 0.7]
for split, sims in [("val", max_sim_val), ("test", max_sim_test)]:
    print(f"
--- {split} ---")
    print(f"  median max-sim : {np.median(sims):.4f}")
    print(f"  mean max-sim   : {np.mean(sims):.4f}")
    for thr in OOD_THRESHOLDS:
        pct = 100 * (sims < thr).mean()
        print(f"  % below {thr}   : {pct:.1f}%")

# ---- 10 most OOD test images ----
test_df = df[test_mask].copy()
test_df["max_train_sim"] = max_sim_test
ood10 = test_df.nsmallest(10, "max_train_sim")[["img_name", "sequence", "max_train_sim"]]
print("
=== 10 most OOD test images (lowest max cosine similarity) ===")
print(ood10.to_string(index=False))


## Section 8 — Report Generation

In [ ]:
import subprocess
import shutil

NOTEBOOK_PATH = ANALYSIS_DIR / "analysis.ipynb"
HTML_PATH     = ANALYSIS_DIR / "report.html"

# --- try nbconvert --to webpdf (requires chromium / pyppeteer) ---
pdf_ok = False
if not REPORT_PATH.exists():
    print("Attempting nbconvert --to webpdf ...")
    r = subprocess.run(
        ["jupyter", "nbconvert", "--to", "webpdf",
         "--output", str(REPORT_PATH), str(NOTEBOOK_PATH)],
        capture_output=True, text=True
    )
    if r.returncode == 0 and REPORT_PATH.exists():
        pdf_ok = True
        print(f"PDF generated at {REPORT_PATH}")
    else:
        print(f"webpdf failed: {r.stderr[-300:]}")

# --- fallback: HTML + weasyprint ---
if not pdf_ok and not REPORT_PATH.exists():
    print("Converting notebook to HTML first ...")
    r = subprocess.run(
        ["jupyter", "nbconvert", "--to", "html",
         "--output", str(HTML_PATH), str(NOTEBOOK_PATH)],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        print(f"HTML saved at {HTML_PATH}")
        print("Converting HTML to PDF via weasyprint ...")
        try:
            from weasyprint import HTML as WP_HTML
            WP_HTML(filename=str(HTML_PATH)).write_pdf(str(REPORT_PATH))
            pdf_ok = True
            print(f"PDF generated at {REPORT_PATH}")
        except Exception as e:
            print(f"weasyprint failed: {e}")
    else:
        print(f"HTML export failed: {r.stderr[-300:]}")

if not pdf_ok:
    print("All PDF methods failed — report is available as HTML only.")
    print(f"HTML report: {HTML_PATH}")
else:
    print(f"Report ready: {REPORT_PATH}")


## Section 9 — Summary Statistics

In [ ]:
from IPython.display import display, Markdown

rows = []
for split in ["train", "val", "test"]:
    mask = df["split"] == split
    n_img  = mask.sum()
    n_seq  = df[mask]["sequence"].nunique()
    if "max_train_sim" in df.columns:
        sim_vals = df.loc[mask & df["max_train_sim"].notna(), "max_train_sim"]
        if len(sim_vals) > 0:
            mean_sim = sim_vals.mean()
            pct_ood  = 100 * (sim_vals < 0.7).mean()
        else:
            mean_sim, pct_ood = float("nan"), float("nan")
    else:
        mean_sim, pct_ood = float("nan"), float("nan")
    rows.append({"split": split, "n_images": n_img, "n_sequences": n_seq,
                 "mean_max_sim": mean_sim, "% OOD (<0.7)": pct_ood})

summary_df = pd.DataFrame(rows)

# Format
def fmt(v):
    if pd.isna(v): return "N/A"
    if isinstance(v, float): return f"{v:.3f}"
    return str(int(v))

md_lines = [
    "| split | n_images | n_sequences | mean_max_sim | % OOD (<0.7) |",
    "|-------|----------|-------------|--------------|---------------|",
]
for _, r in summary_df.iterrows():
    md_lines.append(
        f"| {r['split']} | {fmt(r['n_images'])} | {fmt(r['n_sequences'])} "
        f"| {fmt(r['mean_max_sim'])} | {fmt(r['% OOD (<0.7)']):>6}% |"
    )

md_table = "
".join(md_lines)
display(Markdown(md_table))
print(md_table)

print("
Analysis complete.")
if "ACTUAL_MODEL" not in dir():
    ACTUAL_MODEL = meta.get("model_name", "unknown")
print(f"Model used: {ACTUAL_MODEL}")
